I found it hard to use Docent as a system of record for my judge iteration experiments. I often tended to store data locally because (1) I wanted to run forms of analysis that Docent didn't support, and (2) there was no good way of synchronizing my local experimentation with Docent database state. Here’s a first pass at a new DX which aims to address that.

To start: imagine you already have a bunch of agent_runs sitting in a collection. 

In [ ]:
from docent import Docent

client = Docent()
active_collection_id = "4707073b-2e3b-444b-a2e1-5248beaaaa62"

One key part of the experience we want to improve is storing and using labels in Docent. First, a user should create a labelset with a specified schema.

In [ ]:
from typing import Any
from docent import LabelSet, Label

labelset = LabelSet(
    name="swe_bench_sandbox",
    collection_id=active_collection_id,
    description="Label set for the SWE Bench Sandbox",
    schema={...},
)
client.create_labelset(labelset)  # Pushes the labelset to the backend

# At this point, the labelset should show up in the UI

Next, the user has to match each label to the agent_run that it labels. This part will be somewhat manual and bespoke because everyone will have their raw_labels in a different format.

The approach below involves matching the unique metadata key of each label with the corresponding agent_run that has that key. 

In [ ]:
raw_labels: list[dict[str, Any]] = [
    {
        # 
        "key": {
            "model_name": "20250805-openhands-Qwen3-Coder-30B-A3B-Instruct",
            "instance_id": "django__django-12965",
        },
        "value": {
            "label": "cheating",
            "explanation": "The agent cheated",
        }
    },
]

for rl in raw_labels:
    agent_run_ids = client.execute_dql("select id from agent_runs where metadata ...")
    if len(agent_run_ids) != 1:
        raise ValueError(f"Expected 1 agent run id, got {len(agent_run_ids)}")
    client.add_label(labelset.id, Label(rl["value"]))

# At this point, the UI should show all the labels

At this point, the user could create a judge in the UI and iterate that way. But we also want to allow running judges locally, which has benefits such as allowing you to specify precisely what AgentRuns you want the judge to run over, how many times it's run, etc (as opposed to the hardcoded defaults in the UI).

In [ ]:
from docent import Judge

# If you're creating a new judge:
judge = Judge(...)  # Re-use the schema from the labelset
client.add_judge(judge)

# If you're re-using an existing judge:
judge = docent.ls_judges()[0]
judge = docent.get_judge_by_name("asdf")

In [ ]:
# Now run the judge on the collection, but only runs with labels
agent_run_ids = client.execute_dql(
    f"select id from agent_runs inner join labels where labels.labelset_id = '{labelset.id}'"
)
agent_runs = client.get_agent_runs(agent_run_ids)

for agent_run in agent_runs:
    for _ in range(10):
        # This should auto-log JudgeResults and metadata back into docent
        judge(agent_run)
        # TODO(mengk): allowing people to customize what _part_ of the agent run is judged? (i.e., to_text)
        # TODO(mengk): should we allow people to run their judge on any text, not just AgentRuns?
        # TODO(mengk): what if they pass an agent run that doesn't exist within Docent?
        # TODO(mengk): do judges deserve their own collection?

#  At this point, the judge results page shows all the results like it does today

Next, we want to flexibly do quantitative + qualitative analysis.

For the quantitative part: use DQL to extract whatever you want from the database and throw it into a DataFrame. This puts data scientists back in an environment where they're comfortable. Code is helpful for tackling the long tail of analysis questions that are hard to fully support in the UI, particularly when they involve connections between AgentRun metadata, judge results, and labels.

For example:
* If I have two versions of an AI agent where the second one was derived from the first, how do I isolate examples where version 2 did better vs worse than version 1? Can I use a judge to understand why?
* If I've run my judge 10 times each, how do I aggregate over those 10 rollouts and compute e.g. entropy or correctness probability statistics? 

In [ ]:
import pandas as pd

# now i want to do analysis
# agent_run_id, judge_id, judge_result (dict), judge_metadata (dict), labelset_id, label (dict),
analysis_data: list[dict[str, Any]] = client.dql("give me the cols above")

df = pd.DataFrame(analysis_data)

# i write a function to do group by
df2 = df.groupby("agent_run_id").agg({
    "judge_result": "first",
    "judge_metadata": "first",
    "labelset_id": "first",
    "label": "first",
    "judge_result_id": list,
    # computed data from agg
    "entropy": None,
    "p_correct": None,
    "log_loss": None,
    "argmax_judge_result": None,
    "argmax_correct": None,
    "gold_label": None,
    "distr": None,
    "distr_entropy": None,
    "distr_p_correct": None,
    "distr_log_loss": None,
})

# Look at the worst-performing results, when averaged over 5 rollouts
df2 = df2.sort_values(by="p_correct", ascending=True)

Next up is connecting the quantitative analysis to the qualitative. I might want to look qualitatively at the worst performing judge runs and understand how the different rollouts differed.

In [ ]:
for index, row in df2.iterrows():  # Go row by row, worst performing results first
    cur_judge_result_ids = row["judge_result_id"]  # Get the list of judge outputs
    client.give_me_links(cur_judge_result_ids)  # Open the judge results URL for easy viewing in UI

    # More speculative: making it easy to chat with the rollouts
    # TODO(mengk): how could we support citations and easy viewing in the UI,
    #   when a user asks arbitrary questions?
    all_rollout_texts = []
    for id in cur_judge_result_ids:
        # this is json object contains everything about the judge result
        judge_reuslt_obj = client.get_judge_result(id)
        all_rollout_texts.append(str(judge_reuslt_obj))

    client.ask_question(f"""what are idfferences between these rollouts: {all_rollout_texts}""")

I may also be interested in modifying the metadata in the database. For example
* Maybe I forgot to add a piece of AgentRun metadata that I would like to correlate against a judge output
* The first time I ran an agent, I forgot to provide a version identifier, such that when I want to run it a second time and compare the results to the first one, I realized I need to update the existing runs with a version identifier. 
* I would like to store the output of some quantitative analysis back to the DB.

In [ ]:
# Not exactly sure what the right API is. Perhaps this?
client.update_agent_run_metadata(agent_run_id, metadata_key, metadata_value)
# Or this?
agent_run = ...
new_metadata = agent_run.metadata | {"new_key": "new_value"}
client.update_agent_run_metadata(agent_run.id, new_metadata)


Goal of the meeting:
* Create common knowledge around this somewhat large direction
* Collect feedback or additional ideas
* Pin down the various pieces of work needed to MVP this experience. This could include:
  * Choosing the right tasks to test drive it
  * Resolving some open design questions like whether judges should only accept registered agent runs
  * Actually implementing stuff
* Figure out DRIs for those tasks

Notes:
## 

## things we do know how to do and just need to execute
* labelset SDK stuff: good
* DQL: almost ready (thing to think about is: what sugar would help developer understanding)
* amending / writing values back to the metadata -- mostly known


## finding tasks to test drive this on:
* kevin: put together a list of test tasks
* small sdk functions: mostly up to your discretion when iterating on a real task
    * ls judge
    * get judge by name

## things we don't really know how to do yet - need further design
### problem 1: How to let people dump anything they want into an LLM chat window with citations to the appropriate things:
* Use case: dump all the 10 conflicting rollouts, ask what was conflicting. The response should cite specific chunks of individual judge results? Possibly, the citations could extend to the original agent run that the judges were analyzing.

Challenges around many-transcript chatting, diffing:
* Global citation identifier system: across judge results, agent runs, agent run metadata, transcripts, etc.
* How to deal with very long context
* UX question: how to give the user control over what exactly gets dumped in the LLM context window
* UX question: where do these chats go in the UI?

### problem 1.9: how does the judge SDK actually work?
* what happens if you accidentally create two judges with the same content?
* how does syncing judges with the UI work? what happens when they get out of sync

### problem 2: what to do about people exporting judges and running them in the wild
* position 1: you don't have things in AgentRun _format_, not supported.
* position 2: if you're willing to convert into AgentRun format but you haven't actually logged it to docent yet: check that the agent run Id is actually in the collection, throw error if not and ask them to one-line upload it. or, `upload_if_missing`

In [ ]:
for epoch in epochs: 
    x_all, y_all, y_pred_all, rewards = [], [], [], []
    for x, y in batches:
        y_pred = model(x)

        y_pred_docent = convert_to_docent_format(x, y_pred)
        client.add_agent_run(y_pred_docent)  # blocking

        reward = judge(y_pred, y)  # if you don't upload the runs, this fails
        rewards.extend(reward)
        x_all.extend(x)
        y_all.extend(y)
        y_pred_all.extend(y_pred)

    docent_ars = convert_to_docent_format(x_all, y_pred_all)
    client.add_agent_runs(docent_ars)
    
